# Day 1 Task — World Happiness Report: Exploratory Data Analysis

**Dataset:** World Happiness Report (country-level happiness scores and their drivers)
**Goal:** Understand the dataset, frame a business problem, choose an ML approach, and document initial findings.


## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('world_happiness_report.csv')
df.head()


,Country,Happiness Rank,Happiness Score,Economy,Family,Health,Freedom,Generosity,Corruption,Dystopia,Job Satisfaction,Region
0,Norway,1,7.537,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2.277027,94.6,Western Europe
1,Denmark,2,7.522,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2.313707,93.5,Western Europe
2,Iceland,3,7.504,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2.322715,94.5,Western Europe
3,Switzerland,4,7.494,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2.276716,93.7,Western Europe
4,Finland,5,7.469,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2.430182,91.2,Western Europe


## 2. Dataset Purpose

Each row is a country. The dataset scores each country's overall happiness (`Happiness Score`, roughly 0–10,
based on a Gallup World Poll "ladder" question) alongside six factors researchers use to explain that score:
GDP per capita (`Economy`), social support (`Family`), life expectancy (`Health`), perceived freedom (`Freedom`),
`Generosity`, and perceived absence of corruption (`Corruption`). `Dystopia` is a fixed baseline value added to
the factors, `Job Satisfaction` is an additional survey metric, and `Region` groups countries geographically.

This kind of dataset is used by governments, NGOs, and policy researchers to understand **which factors are most
associated with national wellbeing**, and to benchmark or forecast happiness across countries.


## 3. Business Problem(s)

1. **Driver analysis:** Which factors (economic, social, health, freedom, trust) most strongly influence a
   country's happiness score? This helps policymakers prioritize investment (e.g., healthcare vs. anti-corruption
   reforms) for the biggest wellbeing return.
2. **Happiness prediction:** Given a country's economic and social indicators, can we predict its happiness score?
   Useful for estimating wellbeing in countries/years with incomplete survey data.
3. **Country segmentation:** Can countries be grouped into clusters with similar wellbeing profiles, independent
   of geography? Useful for cross-regional policy benchmarking (e.g., "which non-European countries perform like
   Western Europe?").


## 4. ML Problem Framing

- **Primary framing — Regression.** `Happiness Score` is a continuous numeric variable, so predicting it from the
  factor columns is a **regression** problem (e.g., Linear Regression, Random Forest Regressor). Feature
  coefficients / importances directly answer the "driver analysis" business problem too.
- **Secondary framing — Clustering.** For the segmentation problem, an **unsupervised clustering** approach
  (e.g., K-Means on the factor columns) would group countries by wellbeing profile without using `Region` or
  `Happiness Score` as inputs.
- **Classification** is not the best fit here since there's no natural discrete label to predict — a "happy vs.
  not happy" category would require an arbitrary threshold that throws away information the continuous score
  already provides.


## 5. Target Variable & Key Features

- **Target variable (regression):** `Happiness Score`
- **Key features:** `Economy`, `Family`, `Health`, `Freedom`, `Generosity`, `Corruption`, `Dystopia`,
  `Job Satisfaction`, `Region`
- **Not used as a feature:** `Happiness Rank` (it's derived directly from the target and would leak the answer)


## 6. Basic Exploration with Pandas

### 6.1 Dataset shape

In [2]:
df.shape

(153, 12)

### 6.2 Data types

In [3]:
df.dtypes

Country                 str
Happiness Rank        int64
Happiness Score     float64
Economy             float64
Family              float64
Health              float64
Freedom             float64
Generosity          float64
Corruption          float64
Dystopia            float64
Job Satisfaction    float64
Region                  str
dtype: object

### 6.3 Missing values

In [4]:
missing = df.isnull().sum()
missing[missing > 0]


Job Satisfaction    2
dtype: int64

In [5]:
# Which rows are missing Job Satisfaction?
df[df['Job Satisfaction'].isnull()][['Country', 'Region', 'Happiness Score', 'Job Satisfaction']]


,Country,Region,Happiness Score,Job Satisfaction
59,North Cyprus,Eastern Europe,5.810,NaN
144,South Sudan,Africa,3.591,NaN


### 6.4 Summary statistics

In [6]:
df.describe()

,Happiness Rank,Happiness Score,Economy,Family,Health,Freedom,Generosity,Corruption,Dystopia,Job Satisfaction
count,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000,151.000000
mean,78.169935,5.349281,0.982433,1.186630,0.550117,0.408489,0.245324,0.123179,1.853072,75.209934
std,45.008741,1.134997,0.421901,0.288441,0.237769,0.150744,0.134395,0.102133,0.499490,12.962365
min,1.000000,2.693000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.377914,44.400000
25%,40.000000,4.497000,0.659517,1.041990,0.364509,0.300741,0.153075,0.057070,1.597970,68.950000
50%,78.000000,5.279000,1.064578,1.251826,0.606042,0.437454,0.231503,0.089848,1.832910,78.100000
75%,117.000000,6.098000,1.315175,1.416404,0.719217,0.518631,0.322228,0.153066,2.150801,85.100000
max,155.000000,7.537000,1.870766,1.610574,0.949492,0.658249,0.838075,0.464308,3.117485,95.100000


In [7]:
# Categorical column
df['Region'].value_counts()


Region
Africa            44
Asia-Pacific      43
Latin America     22
Eastern Europe    22
Western Europe    19
North America      2
Europe             1
Name: count, dtype: int64

### 6.5 Correlation with the target

In [8]:
df.corr(numeric_only=True)['Happiness Score'].sort_values(ascending=False)


Happiness Score     1.000000
Job Satisfaction    0.812873
Economy             0.811194
Health              0.780496
Family              0.753815
Freedom             0.576027
Dystopia            0.474300
Corruption          0.435854
Generosity          0.160010
Happiness Rank     -0.992782
Name: Happiness Score, dtype: float64

### 6.6 Happiness by region

In [9]:
df.groupby('Region')['Happiness Score'].agg(['mean', 'count']).sort_values('mean', ascending=False)


,mean,count
Region,,
North America,7.154500,2
Western Europe,6.880474,19
Latin America,5.957818,22
Eastern Europe,5.513091,22
Asia-Pacific,5.358326,43
Africa,4.239500,44
Europe,4.096000,1


## 7. Three Key Observations

1. **Economy and Job Satisfaction are the strongest linear drivers of happiness** — `Job Satisfaction`
   (r ≈ 0.81) and `Economy`/GDP per capita (r ≈ 0.81) correlate with `Happiness Score` more strongly than
   `Health` (r ≈ 0.78) or `Family` (r ≈ 0.75), while `Generosity` is nearly uncorrelated (r ≈ 0.16) — generosity
   alone doesn't predict how happy a country's citizens report being.
2. **Regional inequality is large and structural.** Mean happiness score by region ranges from roughly 6.9 in
   Western Europe down to about 4.2 in Africa, a gap of nearly 3 points on a ~10-point scale — larger than the
   spread you'd expect from any single factor alone, suggesting regional/structural effects compound individual
   drivers.
3. **Missingness is small but not random.** Only `Job Satisfaction` has missing values (2 of 153 rows), and both
   missing rows (North Cyprus, South Sudan) are countries with limited international survey infrastructure —
   this is a hint that missingness itself may correlate with data-collection capacity, not just with happiness.


## 8. Suggested Next Steps

- Handle the 2 missing `Job Satisfaction` values (impute with region median, or drop for modeling).
- Train a baseline Linear Regression and a Random Forest Regressor on the factor columns to predict
  `Happiness Score`; compare R² and RMSE.
- Try K-Means clustering (k selected via elbow/silhouette) on the standardized factor columns to answer the
  segmentation business problem.
